<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Supervisor Pattern – Deep Dive
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M20-Supervisor-Pattern"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 📦 Pakete installieren{ display-mode: "form" }
install_packages([
    'wikipedia',
])

# 1 | Übersicht
---

**M19** zeigte drei Multi-Agent-Patterns auf hohem Niveau: Supervisor, Hierarchisch, Pipeline.  
**M20** vertieft das **Supervisor Pattern** mit realen Tools, Iterations-Schutz und Fehlerbehandlung.

| Thema | M19 | M20 |
|-------|-----|-----|
| Supervisor-Grundkonzept | ✅ Überblick | ✅ Deep Dive |
| Worker Agents | Einfache LLM-Calls | **Mit echten Tools** |
| Iterations-Schutz | ❌ | ✅ **Counter + recursion_limit** |
| Fehlerbehandlung | ❌ | ✅ **Try/Except + Retry-Logik** |
| Supervisor-Begründung | ❌ | ✅ **reasoning-Feld** |
| LangSmith-Debugging | Basis | **Message-Analyse + Tracing** |

**Lernziele:**
- Worker Agents mit LangGraph-kompatiblen Tools aufbauen
- Supervisor mit strukturierter Begründung implementieren
- Iterations-Zähler als Sicherheitsnetz einbauen
- Agenten-Fehler abfangen und elegant behandeln

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Supervisor-Architektur mit Tool-Agents</font> </br></p>

diagram = '''
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nwith_structured_output()"]

    SUP -->|"recherche"| R["🔍 Recherche-Agent\nWikipedia-Tool"]
    SUP -->|"schreiben"| W["✍️ Schreib-Agent\nText-Tools"]
    SUP -->|"code"| C["💻 Code-Agent\nPython-REPL"]
    SUP -->|"FINISH"| END([END])

    R -->|"AIMessage\n(name=Recherche)"| SUP
    W -->|"AIMessage\n(name=Schreiben)"| SUP
    C -->|"AIMessage\n(name=Code)"| SUP

    R --- RT["📚 wikipedia\n@tool"]
    W --- WT["🔢 count_words\n@tool"]
    C --- CT["▶️ python_repl\n@tool"]

    style SUP fill:#FF9800,color:#fff
    style R   fill:#2196F3,color:#fff
    style W   fill:#4CAF50,color:#fff
    style C   fill:#9C27B0,color:#fff
    style RT  fill:#E3F2FD
    style WT  fill:#E8F5E9
    style CT  fill:#F3E5F5
'''
mermaid(diagram, width=820)

# 2 | Worker Agents mit Tools
---

In M19 waren die Worker Agents einfache LLM-Aufrufe ohne Tool-Nutzung.  
In M20 erhält **jeder Worker Agent eigene Tools** – das ist die reale Praxis.

**Tool-Integration in Worker Agents:**

```python
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def wikipedia(query: str) -> str:
    """Sucht Informationen in Wikipedia."""
    ...

# Worker Agent = ReAct-Agent mit Tools
research_agent = create_react_agent(
    llm,
    tools=[wikipedia],
    prompt=SystemMessage("Du bist Recherche-Spezialist...")
)
```

**Warum `create_react_agent`?**

| Eigenschaft | Einfacher LLM-Call | ReAct-Agent mit Tools |
|-------------|-------------------|----------------------|
| Tool-Nutzung | ❌ | ✅ Automatisch |
| Multi-Step | ❌ (ein Schritt) | ✅ (Think → Act → Observe) |
| Fehler-Recovery | ❌ | ✅ Retry möglich |
| Aufwand | Gering | Mittel |

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: ReAct-Agent als Worker</font> </br></p>

diagram = '''
flowchart LR
    IN[/"Aufgabe vom\nSupervisor"/] --> THINK["🤔 Think\nWas brauche\nich?"]
    THINK --> ACT["⚡ Act\nTool aufrufen"]
    ACT --> OBS["👁️ Observe\nErgebnis lesen"]
    OBS -->|"Noch nicht fertig"| THINK
    OBS -->|"Fertig"| OUT[/"AIMessage\n(name=Agent)"/]

    ACT --- T1["📚 wikipedia"]
    ACT --- T2["🌐 web_search"]
    ACT --- T3["▶️ python_repl"]

    style THINK fill:#FF9800,color:#fff
    style ACT   fill:#2196F3,color:#fff
    style OBS   fill:#4CAF50,color:#fff
'''
mermaid(diagram, width=850)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage
import uuid

# ── Konfigurationskonstanten ───────────────────────────────────────────────
MAX_RETRIES = 3   # API-Retry-Versuche bei transienten Fehlern (with_retry)

# ── LLM ───────────────────────────────────────────────────────────────────
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
print("✅ LLM initialisiert")
print(f"   MAX_RETRIES = {MAX_RETRIES}")

In [ ]:
# ── Research Tools ────────────────────────────────────────────────────────
import wikipedia as wiki_lib

@tool
def wikipedia_suche(suchbegriff: str) -> str:
    """Sucht Informationen in Wikipedia. Fuer Fakten, Definitionen, Geschichte.
    Gibt eine kurze Zusammenfassung zurueck.
    TIPP: Englische Begriffe liefern fuer Programmier-Themen bessere Ergebnisse.
    """
    # Englische Wikipedia nutzen: fuer Programmier-Themen vollstaendiger.
    # Deutsche Wikipedia hat viele Fachbegriffe als Redirect oder gar nicht.
    wiki_lib.set_lang("en")
    try:
        return wiki_lib.summary(suchbegriff, sentences=5)
    except wiki_lib.exceptions.DisambiguationError as e:
        # Mehrdeutiger Begriff: erstes konkretes Ergebnis versuchen
        try:
            return wiki_lib.summary(e.options[0], sentences=5)
        except Exception:
            return f"Mehrdeutig. Alternativen: {e.options[:3]}"
    except wiki_lib.exceptions.PageError:
        # Seite nicht gefunden: mit "programming" oder "Python" ergaenzen
        try:
            return wiki_lib.summary(f"{suchbegriff} programming", sentences=5)
        except Exception as e2:
            return f"Artikel nicht gefunden: {suchbegriff}. Fehler: {e2}"
    except Exception as e:
        return f"Wikipedia-Suche fehlgeschlagen: {str(e)}"

@tool
def wichtige_fakten(thema: str) -> str:
    """Extrahiert die wichtigsten Fakten zu einem Thema als Stichpunkte.
    Nutze dies nach wikipedia_suche um die Kernpunkte herauszuarbeiten."""
    return (f"Bitte extrahiere 3-5 Kernfakten zu '{thema}' "
            "aus den bereits gefundenen Informationen.")

# ── Think Tool (Strategische Reflexion) ───────────────────────────────────
@tool
def think(reflection: str) -> str:
    """Strategisches Nachdenken nach einem Tool-Ergebnis.

    Nutze dieses Tool nach jeder wikipedia_suche:
    - Habe ich genug Informationen fuer meine Aufgabe?
    - Welche Aspekte fehlen noch?
    - Welcher naechste Schritt bringt den groessten Mehrwert?

    Args:
        reflection: Deine Analyse des Fortschritts und geplante naechste Schritte
    """
    return f"Reflexion notiert: {reflection}"

# ── Writing Tools ─────────────────────────────────────────────────────────
@tool
def wort_zaehlen(text: str) -> str:
    """Zaehlt Woerter in einem Text. Nuetzlich um Wortlimits einzuhalten."""
    anzahl = len(text.split())
    return f"Der Text hat {anzahl} Woerter."

@tool
def gliederung_erstellen(thema: str) -> str:
    """Erstellt eine strukturierte Gliederung fuer ein Thema."""
    return (
        f"Gliederung fuer '{thema}':\n"
        "1. Einleitung & Definition\n"
        "2. Historische Entwicklung\n"
        "3. Aktuelle Anwendungen\n"
        "4. Vor- und Nachteile\n"
        "5. Fazit & Ausblick"
    )

# ── Code Tools ────────────────────────────────────────────────────────────
@tool
def python_ausfuehren(code: str) -> str:
    """Fuehrt einfachen Python-Code aus und gibt das Ergebnis zurueck.
    Fuer Berechnungen, Demonstrationen und kurze Algorithmen."""
    verboten = ["import os", "import sys", "exec(", "__import__", "open("]
    if any(v in code for v in verboten):
        return "Nicht erlaubt: System-Importe und exec() sind gesperrt."
    try:
        ergebnis = eval(code)
        return f"Ergebnis: {ergebnis}"
    except SyntaxError:
        ns: dict = {}
        exec(compile(code, "<string>", "exec"), ns)
        return f"Ausgefuehrt. Lokale Variablen: {list(ns.keys())}"
    except Exception as e:
        return f"Fehler: {type(e).__name__}: {e}"

@tool
def code_pruefen(code: str) -> str:
    """Prueft Python-Code auf haeufige Probleme und Best-Practice-Verletzungen.
    Gibt Hinweise zur Codequalitaet zurueck."""
    hinweise = []
    if "print(" in code:
        hinweise.append("Tipp: In Produktion logging statt print() verwenden.")
    if len(code.split("\n")) > 30:
        hinweise.append("Tipp: Funktion sehr lang (>30 Zeilen) - Refactoring erwaegen.")
    if "try" not in code and "def " in code:
        hinweise.append("Tipp: Funktionen ohne Fehlerbehandlung koennen abstürzen.")
    return "\n".join(hinweise) if hinweise else "Code sieht solide aus!"

print("✅ Tools definiert:")
print("   wikipedia_suche  (en-Wikipedia + PageError-Handling)")
print("   think            (strategische Reflexion nach Tool-Calls)")
print("   wichtige_fakten  (fuer Aufgabe 1 - Anti-Pattern-Demo)")
print("   wort_zaehlen, gliederung_erstellen")
print("   python_ausfuehren, code_pruefen")

In [ ]:
# ── Spezialisierte Worker Agents ─────────────────────────────────────────
# Jeder Agent hat einen klaren Fokus und nur die Tools, die er braucht.
# ACHTUNG: wichtige_fakten ist NICHT im recherche_agent!
#   Das Tool gibt eine Aufgaben-Beschreibung zurück, kein Ergebnis.
#   Der Agent würde es als neue Aufgabe interpretieren → extra Schritte
#   → recursion_limit überschritten. (Lernhinweis für Aufgabe 1)

# 1. Recherche-Agent – wikipedia_suche + think (strategische Reflexion)
#    think: Nach jeder Suche reflektiert der Agent ob er genug Infos hat.
#    Verhindert unnötige Folgesuchen und verbessert die Fokussierung.
recherche_agent = create_react_agent(
    llm,
    tools=[wikipedia_suche, think],
    prompt=SystemMessage(
        "Du bist Recherche-Spezialist. "
        "Nutze wikipedia_suche um Fakten zu recherchieren. "
        "Nutze think nach jeder Suche: Hast du genug? Was fehlt noch? "
        "Fasse die wichtigsten Informationen prägnant zusammen. "
        "Antworte auf Deutsch."
    )
)

# 2. Schreib-Agent – Gliederung und Wort-Zählung
schreib_agent = create_react_agent(
    llm,
    tools=[wort_zaehlen, gliederung_erstellen],
    prompt=SystemMessage(
        "Du bist Schreib-Spezialist. Erstelle gut strukturierte Texte. "
        "Nutze gliederung_erstellen für Struktur und wort_zaehlen "
        "zum Einhalten von Wortlimits. Klare Sprache. Deutsch."
    )
)

# 3. Code-Agent – Python ausführen und prüfen
code_agent = create_react_agent(
    llm,
    tools=[python_ausfuehren, code_pruefen],
    prompt=SystemMessage(
        "Du bist Code-Spezialist. Schreibe sauberen, kommentierten Python-Code. "
        "Teste ihn mit python_ausfuehren und prüfe ihn mit code_pruefen. "
        "Erkläre deinen Code auf Deutsch."
    )
)

print("✅ Worker Agents erstellt:")
print("   recherche_agent → [wikipedia_suche, think]")
print("   schreib_agent   → [wort_zaehlen, gliederung_erstellen]")
print("   code_agent      → [python_ausfuehren, code_pruefen]")

# 3 | Supervisor mit Begründung & Iterations-Schutz
---

> **📚 Modell-Auswahl: Supervisor vs. Worker**
>
> In diesem Modul wird die Modell-Rollentrennung erstmals in der Praxis eingesetzt:
> - **Supervisor** (`o3`): trifft Routing-Entscheidungen — starkes Reasoning erforderlich
> - **Worker** (`gpt-4o-mini`): erzeugt Inhalte — Geschwindigkeit und Kosten im Vordergrund
>
> ⚠️ `o3` akzeptiert **keinen** `temperature`-Parameter.
>
> → [Modell-Auswahl Guide](../docs/frameworks/Modell_Auswahl_Guide.md)


<p><font color='black' size="5">
Erweiterungen gegenüber M19
</font></p>

**1. Begründungs-Feld im Supervisor:**
Der Supervisor erklärt jede Entscheidung – unverzichtbar für Debugging.

```python
class SupervisorEntscheidung(BaseModel):
    naechster: Literal["recherche", "schreiben", "code", "FINISH"]
    begruendung: str  # ← NEU: Warum dieser Agent?
```

**2. Iterations-Schutz im State:**
Ein Zähler verhindert endlose Schleifen – unabhängig vom `recursion_limit`.

```python
class SupervisorState(TypedDict):
    messages:    Annotated[list, add_messages]
    aufgabe:     str
    naechster:   str
    iterationen: int  # ← NEU: Schutz vor Endlosschleifen
    max_iter:    int  # ← NEU: Konfigurierbare Grenze
```

**3. Sicherheitsnetz im Supervisor-Node:**
```python
def supervisor_node(state):
    if state["iterationen"] >= state["max_iter"]:
        return {"naechster": "FINISH"}  # Zwangsbeendigung
    ...
```

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Supervisor mit Iterations-Schutz</font> </br></p>

diagram = '''
flowchart TD
    START([START]) --> SUP

    SUP{"🎯 Supervisor\nIterationen < max_iter?"}
    SUP -->|"JA: naechster Agent"| ROUTE["🔀 Router\nsupervisor_router()"]
    SUP -->|"NEIN: Zwang-FINISH"| END

    ROUTE -->|"recherche"| R["🔍 Recherche"]
    ROUTE -->|"schreiben"| W["✍️ Schreiben"]
    ROUTE -->|"code"| C["💻 Code"]
    ROUTE -->|"FINISH"| END([END])

    R & W & C -->|"iterationen++"| SUP

    note1["🛡️ Schutzebene 1:\niterations Counter"]
    note2["🛡️ Schutzebene 2:\nrecursion_limit"]

    style SUP   fill:#FF9800,color:#fff
    style ROUTE fill:#607D8B,color:#fff
    style R     fill:#2196F3,color:#fff
    style W     fill:#4CAF50,color:#fff
    style C     fill:#9C27B0,color:#fff
    style note1 fill:#FFF9C4
    style note2 fill:#FFF9C4
'''
mermaid(diagram, width=820)

In [ ]:
# ── State mit Iterations-Schutz ───────────────────────────────────────────
class SupervisorState(TypedDict):
    messages:    Annotated[list, add_messages]  # Gesamte Nachrichten-Historie
    aufgabe:     str    # Original-Aufgabe (unveraenderlich)
    naechster:   str    # Routing-Entscheidung des Supervisors
    iterationen: int    # Wie oft haben Worker Agents gearbeitet?
    max_iter:    int    # Maximale Anzahl Worker-Runs

# ── Pydantic-Modell mit Begruendungs-Feld ─────────────────────────────────
class SupervisorEntscheidung(BaseModel):
    """Strukturierte Routing-Entscheidung des Supervisors."""
    naechster: Literal["recherche", "schreiben", "code", "FINISH"] = Field(
        description="Naechster Agent oder FINISH wenn die Aufgabe vollstaendig ist."
    )
    begruendung: str = Field(
        description="Kurze Begruendung fuer diese Entscheidung (1-2 Saetze)."
    )

# o3 akzeptiert keinen temperature-Parameter (API-Fehler bei temperature=0.0)
# .with_retry(): schützt vor transienten API-Fehlern (Netzwerk, Rate-Limits)
supervisor_llm = (
    init_chat_model("openai:o3")
    .with_structured_output(SupervisorEntscheidung)
    .with_retry(stop_after_attempt=MAX_RETRIES)
)

# Supervisor-Prompt: XML-Sektionen + History-Lese-Anleitung + Hard Limits
SUPERVISOR_SYSTEM = (
    "Du koordinierst ein Experten-Team.\n\n"
    "<Team>\n"
    "- recherche: Recherchiert Fakten (Wikipedia). Zuerst aufrufen.\n"
    "- schreiben: Erstellt strukturierte Texte. Nach der Recherche.\n"
    "- code:      Schreibt Python-Code. Bei Code-Aufgaben.\n"
    "</Team>\n\n"
    "<Workflow>\n"
    "Typische Reihenfolgen:\n"
    "  Frage/Report:  recherche -> schreiben -> FINISH\n"
    "  Code-Aufgabe:  recherche -> code -> FINISH\n"
    "  Nur Code:      code -> FINISH\n\n"
    "Wie du die Nachrichtenhistorie liest:\n"
    "  name=Recherche -> recherche war bereits aktiv\n"
    "  name=Schreiben -> schreiben war bereits aktiv\n"
    "  name=Code      -> code war bereits aktiv\n"
    "  name=System + Fehler -> Agent fehlgeschlagen (NICHT nochmal schicken!)\n"
    "</Workflow>\n\n"
    "<Hard Limits>\n"
    "Agent-Budget: maximal 2 Agenten-Aufrufe pro Aufgabe.\n"
    "Sofort FINISH wenn:\n"
    "- Alle benoetigen Agenten waren aktiv (Erfolg oder Fehler)\n"
    "- Die Aufgabe vollstaendig beantwortet ist\n"
    "</Hard Limits>\n\n"
    "<Rules>\n"
    "1. Pruefe die Message-Namen VOR jeder Entscheidung.\n"
    "2. Jeden Agenten maximal EINMAL schicken (Erfolg ODER Fehler).\n"
    "3. Wenn alle benoetigen Agenten dran waren: FINISH.\n"
    "4. Bei Fehlern: Naechsten Agenten nehmen, nicht wiederholen.\n"
    "</Rules>"
)

print("✅ State und Supervisor-LLM konfiguriert")
print(f"   supervisor_llm: o3 + with_structured_output + with_retry(stop_after_attempt={MAX_RETRIES})")

In [ ]:
# ── Supervisor Node ───────────────────────────────────────────────────────
def supervisor_node(state: SupervisorState) -> dict:
    """Liest die Situation, entscheidet wer als nächstes arbeitet."""
    if state["iterationen"] >= state["max_iter"]:
        mprint(f"⚠️ **Iterations-Limit ({state['max_iter']}). Erzwinge FINISH.**")
        return {"naechster": "FINISH"}

    entscheidung = supervisor_llm.invoke([
        SystemMessage(SUPERVISOR_SYSTEM),
        HumanMessage(f"Aufgabe: {state['aufgabe']}"),
        *state["messages"],
    ])
    mprint(f"**👔 Supervisor →** `{entscheidung.naechster}`  _{entscheidung.begruendung}_")
    return {"naechster": entscheidung.naechster}


# ── Worker Node Factory ────────────────────────────────────────────────────
# create_react_agent nutzt intern einen eigenen LangGraph-Graphen.
# Jeder ReAct-Schritt zählt als Graph-Step:
#   1 Tool-Aufruf = Think(1) + Act(1) + Observe(1) = 3 Steps
#   2 Tool-Aufrufe = 5 Steps  |  3 Tool-Aufrufe = 7 Steps
# Mit einem Tool pro Agent reicht limit=15 sicher aus.
INNER_AGENT_CFG = {"recursion_limit": 15}

def make_worker_node(agent, name: str):
    """Worker-Node für LangGraph.

    Übergibt dem inneren ReAct-Agenten ein eigenes recursion_limit,
    damit er genug Schritte für Tool-Calls hat.
    """
    def worker_node(state: SupervisorState) -> dict:
        mprint(f"\n**🤖 {name}** arbeitet...")
        kontext = f"Aufgabe: {state['aufgabe']}"
        if state["messages"]:
            bisherige = "\n".join(
                f"{getattr(m, 'name', 'User')}: {m.content[:300]}"
                for m in state["messages"]
            )
            kontext += f"\n\nBisherige Arbeit des Teams:\n{bisherige}"

        result = agent.invoke(
            {"messages": [HumanMessage(kontext)]},
            config=INNER_AGENT_CFG,
        )
        letzte_msg = result["messages"][-1]
        return {
            "messages":    [AIMessage(content=letzte_msg.content, name=name)],
            "iterationen": state["iterationen"] + 1,
        }
    worker_node.__name__ = f"{name}_node"
    return worker_node


# ── Router ─────────────────────────────────────────────────────────────────
def supervisor_router(state: SupervisorState) -> str:
    naechster = state.get("naechster", "FINISH")
    return "__end__" if naechster == "FINISH" else naechster


recherche_node = make_worker_node(recherche_agent, "Recherche")
schreiben_node = make_worker_node(schreib_agent,   "Schreiben")
code_node      = make_worker_node(code_agent,      "Code")

print("✅ Supervisor-Node, Worker-Nodes und Router bereit")
print(f"   INNER_AGENT_CFG: recursion_limit={INNER_AGENT_CFG['recursion_limit']}")

# 4 | Graph aufbauen
---


<p><font color='black' size="5">
COMPILE – Supervisor-Graph assemblieren
</font></p>

**Zwei Schutzebenen gegen Endlosschleifen:**

| Schutz | Mechanismus | Wann greift er? |
|--------|------------|----------------|
| `iterationen >= max_iter` | Feld im State | Supervisor prüft vor jeder Entscheidung |
| `recursion_limit` | LangGraph intern | Maximale Graph-Schritt-Anzahl |

```python
# Beide Grenzen konfigurieren:
graph.invoke(input, config={"recursion_limit": 30})
#                             ↑ je 2 Schritte pro Agent-Runde
```

In [ ]:
# ── COMPILE ───────────────────────────────────────────────────────────────
builder = StateGraph(SupervisorState)

# Nodes hinzufügen
builder.add_node("supervisor", supervisor_node)
builder.add_node("recherche",  recherche_node)
builder.add_node("schreiben",  schreiben_node)
builder.add_node("code",       code_node)

# Einstiegspunkt
builder.add_edge(START, "supervisor")

# Bedingtes Routing vom Supervisor
builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "recherche": "recherche",
        "schreiben": "schreiben",
        "code":      "code",
        "__end__":   END,           # Explizite path_map – korrekte Visualisierung
    }
)

# Nach jedem Worker zurück zum Supervisor (Sternförmige Topologie)
builder.add_edge("recherche", "supervisor")
builder.add_edge("schreiben", "supervisor")
builder.add_edge("code",      "supervisor")

# Kompilieren
supervisor_graph = builder.compile()
print("✅ Supervisor-Graph kompiliert")

In [ ]:
# ── VIZ ───────────────────────────────────────────────────────────────────
try:
    display(IPImage(supervisor_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(Graph-Visualisierung nicht verfügbar: {e})")

In [ ]:
# ── TESTS: Task 1 – Recherche + Schreiben ─────────────────────────────────
run_cfg = {"run_name": "M20-Test1", "tags": ["m19", "supervisor"],
           "recursion_limit": 30}

aufgabe1 = "Erkläre die Grundprinzipien von Large Language Models und schreibe einen strukturierten Überblick."

result1 = supervisor_graph.invoke(
    {
        "messages":    [],
        "aufgabe":     aufgabe1,
        "naechster":   "",
        "iterationen": 0,
        "max_iter":    5,
    },
    config=run_cfg
)

mprint("## 🎯 Ergebnis: Recherche + Schreiben")
mprint(f"**Aufgabe:** {aufgabe1}\n")
mprint(f"**Iterationen:** {result1['iterationen']} von max. 5\n")
mprint("---")
for msg in result1["messages"]:
    agent = getattr(msg, "name", None) or "System"
    mprint(f"**[{agent}]**\n{msg.content}\n---")

In [ ]:
# ── TESTS: Task 2 – Recherche + Code ─────────────────────────────────────
aufgabe2 = (
    "Recherchiere die Fibonacci-Folge und schreibe dann Python-Code "
    "für eine effiziente Implementierung mit Caching."
)

result2 = supervisor_graph.invoke(
    {
        "messages":    [],
        "aufgabe":     aufgabe2,
        "naechster":   "",
        "iterationen": 0,
        "max_iter":    5,
    },
    config={"run_name": "M20-Test2", "tags": ["m19"], "recursion_limit": 30}
)

mprint("## 🎯 Ergebnis: Recherche + Code")
mprint(f"**Aufgabe:** {aufgabe2}\n")
mprint(f"**Iterationen:** {result2['iterationen']}\n")
mprint("---")
letzte = result2["messages"][-1]
mprint(f"**Finale Antwort [{getattr(letzte, 'name', 'Agent')}]:**\n{letzte.content}")

# 5 | Fehlerbehandlung im Supervisor-System
---


<p><font color='black' size="5">
Warum Fehlerbehandlung notwendig ist
</font></p>

In produktiven Multi-Agent-Systemen können Worker Agents fehlschlagen:
- Tool wirft eine Exception
- LLM-Antwort ist leer oder ungültig
- Netzwerkfehler bei externen APIs

**Strategie: Graceful Degradation**
Statt den gesamten Graph abzubrechen, fängt ein fehlertoleranter Worker-Node
Ausnahmen ab und gibt eine Fehler-Nachricht zurück.
Der Supervisor entscheidet dann, ob er es erneut versucht oder fortfährt.

```python
def fehlertoleranter_worker(state):
    try:
        result = agent.invoke(...)
        return {"messages": [AIMessage(...)]}
    except Exception as e:
        # Fehler als Nachricht → Supervisor kann reagieren
        return {"messages": [AIMessage(
            content=f"❌ Fehler: {e}. Bitte alternative Strategie wählen.",
            name="System"
        )]}
```

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Fehlerbehandlung im Worker</font> </br></p>

diagram = '''
flowchart TD
    SUP["🎯 Supervisor"] --> WORKER["🤖 Worker Agent"]

    WORKER --> TRY{{"Try: Tool aufrufen"}}
    TRY -->|"Erfolg"| OK["✅ AIMessage\n(name=Agent)"]
    TRY -->|"Exception"| ERR["❌ Fehler-Nachricht\n(name=System)"]

    OK  --> SUP2["🎯 Supervisor\nWeiter planen"]
    ERR --> SUP2

    SUP2 -->|"Erfolg"| NEXT["➡️ Nächster Agent"]
    SUP2 -->|"Fehler"| RETRY["🔄 Retry oder\nanderer Agent"]
    SUP2 -->|"Aufgabe lösbar?"| FIN["🏁 FINISH"]

    style TRY   fill:#FF9800,color:#fff
    style OK    fill:#4CAF50,color:#fff
    style ERR   fill:#f44336,color:#fff
    style SUP   fill:#FF9800,color:#fff
    style SUP2  fill:#FF9800,color:#fff
'''
mermaid(diagram, width=720)

In [ ]:
# ── Fehlertolerante Worker Node Factory ──────────────────────────────────
def make_resilient_worker(agent, name: str, max_retries: int = 1):
    """Fehlertoleranter Worker-Node mit Retry-Logik.

    Fängt zwei Fehlerklassen ab:
    1. Python-Exceptions (Netzwerk, API, Tool-Fehler)
    2. "Sorry, need more steps" – LangGraph-Meldung bei überschrittenem
       recursion_limit. Wird durch INNER_AGENT_CFG verhindert, aber
       als Fallback zusätzlich erkannt.
    """
    def resilient_node(state: SupervisorState) -> dict:
        mprint(f"\n**🤖 {name}** (resilient) arbeitet...")
        kontext = f"Aufgabe: {state['aufgabe']}"
        if state["messages"]:
            bisherige = "\n".join(
                f"{getattr(m, 'name', 'User')}: {m.content[:300]}"
                for m in state["messages"]
            )
            kontext += f"\n\nBisherige Arbeit:\n{bisherige}"

        for versuch in range(1, max_retries + 2):
            try:
                result = agent.invoke(
                    {"messages": [HumanMessage(kontext)]},
                    config=INNER_AGENT_CFG,
                )
                letzte = result["messages"][-1]
                # Weiches Versagen: "Sorry, need more steps"
                if ("sorry" in letzte.content.lower()
                        and "more steps" in letzte.content.lower()):
                    raise RuntimeError(
                        "Inneres recursion_limit erreicht. "
                        f"Erhöhe INNER_AGENT_CFG['recursion_limit'] "
                        f"(aktuell: {INNER_AGENT_CFG['recursion_limit']})."
                    )
                return {
                    "messages":    [AIMessage(content=letzte.content, name=name)],
                    "iterationen": state["iterationen"] + 1,
                }
            except Exception as e:
                if versuch <= max_retries:
                    mprint(f"  ⚠️ Versuch {versuch} fehlgeschlagen: {e}. Wiederhole...")
                else:
                    fehler_msg = (
                        f"❌ {name} konnte Aufgabe nicht erfüllen "
                        f"nach {max_retries + 1} Versuchen: {e}"
                    )
                    return {
                        "messages":    [AIMessage(content=fehler_msg, name="System")],
                        "iterationen": state["iterationen"] + 1,
                    }
        return {"messages": [], "iterationen": state["iterationen"] + 1}

    resilient_node.__name__ = f"{name}_resilient_node"
    return resilient_node


res_recherche = make_resilient_worker(recherche_agent, "Recherche", max_retries=1)
res_schreiben = make_resilient_worker(schreib_agent,   "Schreiben", max_retries=1)
res_code      = make_resilient_worker(code_agent,      "Code",      max_retries=1)

print("✅ Resiliente Worker Nodes erstellt")
print(f"   max_retries=1 | recursion_limit={INNER_AGENT_CFG['recursion_limit']}")

In [ ]:
# ── COMPILE: Resilient Graph ──────────────────────────────────────────────
builder_r = StateGraph(SupervisorState)

builder_r.add_node("supervisor", supervisor_node)
builder_r.add_node("recherche",  res_recherche)
builder_r.add_node("schreiben",  res_schreiben)
builder_r.add_node("code",       res_code)

builder_r.add_edge(START, "supervisor")

builder_r.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {"recherche": "recherche", "schreiben": "schreiben",
     "code": "code", "__end__": END}
)

builder_r.add_edge("recherche", "supervisor")
builder_r.add_edge("schreiben", "supervisor")
builder_r.add_edge("code",      "supervisor")

resilient_graph = builder_r.compile()
print("✅ Resilient-Graph kompiliert")

In [ ]:
# ── VIZ: Resilient Graph ──────────────────────────────────────────────────
try:
    display(IPImage(resilient_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(Graph-Visualisierung nicht verfügbar: {e})")

In [ ]:
# ── TEST: Resilient Graph mit normalem Task ───────────────────────────────
aufgabe3 = "Recherchiere Python-Decorators und erstelle ein Code-Beispiel."

result3 = resilient_graph.invoke(
    {
        "messages":    [],
        "aufgabe":     aufgabe3,
        "naechster":   "",
        "iterationen": 0,
        "max_iter":    6,
    },
    config={"run_name": "M20-Resilient", "tags": ["m19", "resilient"],
            "recursion_limit": 40}
)

mprint("## 🎯 Resilient Graph – Ergebnis")
mprint(f"**Iterationen:** {result3['iterationen']}\n")
for msg in result3["messages"]:
    agent = getattr(msg, "name", None) or "System"
    if agent == "System":
        mprint(f"**⚠️ [{agent}]** {msg.content}")
    else:
        mprint(f"**[{agent}]** {msg.content[:300]}...\n---")

# 6 | Debugging & Best Practices
---


<p><font color='black' size="5">
Nachrichten analysieren
</font></p>

Nach einem Graph-Lauf lässt sich die Zusammenarbeit der Agents detailliert auswerten.

In [ ]:
# ── Nachrichten-Analyse nach dem Graph-Lauf ──────────────────────────────
def analysiere_lauf(result: dict, title: str = "Lauf-Analyse"):
    """Analysiert einen Graph-Lauf: Agent-Beiträge, Nachrichten-Längen."""
    messages = result["messages"]
    agent_beitraege = {}
    for msg in messages:
        name = getattr(msg, "name", None) or "Unbekannt"
        agent_beitraege[name] = agent_beitraege.get(name, 0) + 1

    mprint(f"## 📊 {title}")
    mprint(f"**Gesamt Nachrichten:** {len(messages)}")
    mprint(f"**Iterationen:** {result.get('iterationen', '–')}")
    mprint("\n**Agent-Beiträge:**")
    for agent, count in sorted(agent_beitraege.items()):
        mprint(f"  - `{agent}`: {count}x")

    # Längste Nachricht
    laengste = max(messages, key=lambda m: len(m.content))
    mprint(f"\n**Längste Nachricht:** `{getattr(laengste, 'name', '?')}` "
           f"({len(laengste.content)} Zeichen)")


# Analyse für beide Test-Läufe
analysiere_lauf(result1, "Test 1: Recherche + Schreiben")
print()
analysiere_lauf(result2, "Test 2: Recherche + Code")


<p><font color='black' size="5">
Best Practices: Supervisor Pattern
</font></p>

| Aspekt | Empfehlung |
|--------|------------|
| **Agent-Anzahl** | 3-5 Agents; mehr → Hierarchisches Pattern (M30) |
| **Supervisor-Prompt** | Klare Reihenfolge-Empfehlungen angeben |
| **Iterations-Schutz** | Immer `max_iter` IM State + `recursion_limit` in Config |
| **Nachrichten-Kontext** | Zu lange Histories → Kosten steigen, Quality sinkt → trimmen |
| **Tool-Fehler** | Resiliente Nodes mit Try/Except + Fehlernachricht |
| **Begründungs-Feld** | `begruendung` im Pydantic-Modell für Debugging |
| **LangSmith** | `run_name` + `tags` für alle Runs setzen |

**Häufige Fehler**

| Fehler | Ursache | Lösung |
|--------|---------|--------|
| Endlosschleife | Supervisor kommt nicht zu FINISH | `max_iter` + `recursion_limit` |
| Floating Nodes | Router gibt `END`-Konstante zurück | `"__end__"` (String) + `path_map` |
| Leere Antworten | Agent erhält zu wenig Kontext | Aufgaben-Feld im State mitgeben |
| Hohe Kosten | Zu viele Supervisor-Calls | `with_structured_output` + gute Prompts |
| Falsche Reihenfolge | Supervisor-Prompt unklar | Konkrete Reihenfolgen im Prompt |

**Supervisor vs. Pipeline – Entscheidungshilfe**

```
Aufgabe hat immer dieselben Schritte?
  → JA: Pipeline (M19)
  → NEIN: Schritte hängen von Zwischen-Ergebnissen ab?
      → JA: Supervisor (M20)
      → NEIN: Brauche ich mehrere Teams?
          → JA: Hierarchisch (M30)
```

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M20-Supervisor-Pattern", limit=3, show_steps=True)

# A | Aufgabe
---

Die Aufgabenstellungen unten bieten Anregungen. Sie können aber auch gerne eine andere Herausforderung angehen.



<p><font color='black' size="5">
Aufgabe 1: Critique Agent hinzufügen
</font></p>

Erweitere das Supervisor-System um einen **Critique-Agent**, der Output bewertet:

```python
class KritikBewertung(BaseModel):
    punktzahl:  int  # 1-10
    feedback:   str
    genehmigt:  bool

@tool
def bewertung_speichern(bewertung: str) -> str:
    """Speichert eine Bewertung für den Bericht."""
    ...
```

- Critique-Agent prüft den Schreib-Agent-Output
- Bei `genehmigt=False` → Supervisor schickt Schreib-Agent erneut
- Maximal 2 Kritik-Runden



<p><font color='black' size="5">
Aufgabe 2: Nachrichten-Trimming
</font></p>

Lange Agent-Historien erhöhen Kosten und Latenz. Implementiere Trimming:

```python
from langchain_core.messages import trim_messages

def supervisor_node_mit_trimming(state: SupervisorState) -> dict:
    # Nur die letzten N Tokens an den Supervisor senden
    gekuerzte_messages = trim_messages(
        state["messages"],
        max_tokens=2000,
        strategy="last",
        token_counter=llm,
    )
    entscheidung = supervisor_llm.invoke([...])
    ...
```



<p><font color='black' size="5">
Aufgabe 3: Parallele Worker-Ausführung
</font></p>

Nutze LangGraphs `Send()` API um mehrere Agents gleichzeitig zu starten:

```python
from langgraph.types import Send

def supervisor_parallel(state: SupervisorState):
    """Startet Recherche und Code-Agent gleichzeitig."""
    return [
        Send("recherche", state),
        Send("code",      state),
    ]
```

**Bonus:** Kombiniere mit HITL aus M18 – Mensch genehmigt den finalen Bericht  
und Checkpointing aus M15 – Berichte persistent in SQLite speichern.